In [1]:
import numpy as np
import xarray as xr

In [2]:
def parse_dataset(path:str):
    _data = np.loadtxt(path, unpack=True)
    wave_number, _counts = _data[-2], _data[-1]
    num_coords = len(_data) - 2
    coords = [(_data[idx]-_data[idx].min()).astype(int) for idx in range(num_coords)]
    unique_coords = [np.unique(coord, return_inverse=True) for coord in coords]
    unique_coords += [np.unique(wave_number, return_inverse=True)] # [(X_value, X_idx), ... (WN_value, WN_idx)]
    idxs, values = [coord[1] for coord in unique_coords], [coord[0] for coord in unique_coords]
    counts_shape = tuple([len(coord[0]) for coord in unique_coords])
    counts = np.zeros(counts_shape, dtype = _counts.dtype)

    for z in zip(*idxs, _counts):
        counts[*z[:-1]] = z[-1]

    dimension_names = [f'X_{i}' for i in range(len(values) -1)] + ['wave_number']

    counts = xr.DataArray(counts, coords=values, dims=dimension_names)
    counts.wave_number.attrs['units'] = 'cm^-1'
    return counts

In [3]:
parse_raw = True
data_files = ['raw_data/Mixed_8x8.txt', 'raw_data/map_c_100x10.txt', 'raw_data/map_c_50x20_mixed2.txt', 'raw_data/DepthMapHR100x5p1s1acc.txt']
system_type = ['SiC+Graphene', 'SiC', 'SiC+Graphene', 'SiC']
if parse_raw:
    datasets = [parse_dataset(path) for path in data_files]
    for i, dataset in enumerate(datasets):
        dataset.attrs['system_type'] = system_type[i]
        #Save to netcdf
        dims = dataset.dims[:-1]
        fname = f'../data/{system_type[i]}_'
        for dim_idx in range(len(dims) -1):
            fname += f'{len(dataset[dims[dim_idx]])}x'
        fname += f'{len(dataset[dims[-1]])}.nc'
        dataset.to_netcdf(fname)

In [4]:
example_dataset1 = parse_dataset('raw_data/DepthMapHR100x5p1s1acc.txt')
print(example_dataset1.shape)
print(example_dataset1.dtype)
print(example_dataset1)

(19, 10, 3, 1015)
float64
<xarray.DataArray (X_0: 19, X_1: 10, X_2: 3, wave_number: 1015)>
array([[[[ 74.983772,  91.177582,  85.314659, ...,  29.386446,
           20.207445,  16.533365],
         [ 73.513504,  73.530304,  73.547119, ...,  27.549793,
           33.06673 ,  22.044485],
         [ 67.632423,  70.589096,  63.250523, ...,  18.366529,
           20.207445,  22.044485]],

        [[ 92.627014,  80.883339,  85.314659, ...,  20.203182,
           31.229689,  25.718567],
         [ 76.454041,  98.530609,  79.430893, ...,  27.549793,
           29.392649,  36.74081 ],
         [ 58.810799,  69.118492,  57.366753, ...,  25.71314 ,
           20.207445,  23.881527]],

        [[ 82.335121,  98.530609,  88.256546, ...,  36.733059,
           25.718567,  18.370405],
         [ 77.924309,  73.530304,  83.843719, ...,  27.549793,
           34.90377 ,  25.718567],
         [ 54.39999 ,  70.589096,  58.837696, ...,  22.039835,
           23.881527,  23.881527]],
...
        [[ 74.9837

In [5]:
example_dataset2 = parse_dataset('raw_data/map_c_100x10.txt')
print(example_dataset2.shape)
print(example_dataset2.dtype)
print(example_dataset2)

(100, 10, 1015)
float64
<xarray.DataArray (X_0: 100, X_1: 10, wave_number: 1015)>
array([[[ 82.334389, 129.412186,  88.25576 , ...,  29.386202,
          58.784805,  47.762657],
        [ 94.096443,  94.117958, 105.906914, ...,  34.896114,
          55.110756,  86.340187],
        [ 69.102074,  94.117958,  92.668549, ...,  47.752579,
          40.414555,  36.740505],
        ...,
        [ 60.280533,  89.706177, 102.96505 , ...,  44.079304,
          80.829109,  69.806961],
        [ 82.334389,  79.412025,  80.901115, ...,  62.445679,
          75.318031,  36.740505],
        [ 86.745163,  89.706177, 113.261559, ...,  56.935768,
          73.48101 ,  69.806961]],

       [[ 92.62619 , 108.823891, 101.494125, ...,  42.242664,
          67.969933,  42.251579],
        [ 79.393875,  82.35321 ,  95.610405, ...,  49.589218,
          11.022151,  49.599682],
        [ 69.102074,  98.529739, 108.84877 , ...,  64.282318,
          40.414555,  66.132904],
...
        [ 85.274902,  75.000244,  8

In [6]:
example_dataset_sum = xr.DataArray.combine_first(example_dataset1, example_dataset2)
print(example_dataset_sum.shape)
print(example_dataset_sum.dtype)

(100, 10, 3, 2030)
float64


In [7]:
example_dataset_sum

<xarray.DataArray (X_0: 100, X_1: 10, X_2: 3, wave_number: 2030)>
array([[[[ 82.334389,  74.983772, 129.412186, ...,  20.207445,
           47.762657,  16.533365],
         [ 82.334389,  73.513504, 129.412186, ...,  33.06673 ,
           47.762657,  22.044485],
         [ 82.334389,  67.632423, 129.412186, ...,  20.207445,
           47.762657,  22.044485]],

        [[ 94.096443,  92.627014,  94.117958, ...,  31.229689,
           86.340187,  25.718567],
         [ 94.096443,  76.454041,  94.117958, ...,  29.392649,
           86.340187,  36.74081 ],
         [ 94.096443,  58.810799,  94.117958, ...,  20.207445,
           86.340187,  23.881527]],

        [[ 69.102074,  82.335121,  94.117958, ...,  25.718567,
           36.740505,  18.370405],
         [ 69.102074,  77.924309,  94.117958, ...,  34.90377 ,
           36.740505,  25.718567],
         [ 69.102074,  54.39999 ,  94.117958, ...,  23.881527,
           36.740505,  23.881527]],
...
        [[ 80.864136,        nan, 101.470924, ...,        nan,
           69.806961,        nan],
         [ 80.864136,        nan, 101.470924, ...,        nan,
           69.806961,        nan],
         [ 80.864136,        nan, 101.470924, ...,        nan,
           69.806961,        nan]],

        [[ 69.102074,        nan,  86.764992, ...,        nan,
           62.458858,        nan],
         [ 69.102074,        nan,  86.764992, ...,        nan,
           62.458858,        nan],
         [ 69.102074,        nan,  86.764992, ...,        nan,
           62.458858,        nan]],

        [[ 82.334389,        nan, 105.882698, ...,        nan,
           53.273731,        nan],
         [ 82.334389,        nan, 105.882698, ...,        nan,
           53.273731,        nan],
         [ 82.334389,        nan, 105.882698, ...,        nan,
           53.273731,        nan]]]])
Coordinates:
  * X_0          (X_0) int64 0 1 2 3 4 5 6 7 8 9 ... 91 92 93 94 95 96 97 98 99
  * X_1          (X_1) int64 0 1 2 3 4 5 6 7 8 9
  * X_2          (X_2) int64 0 2 4
  * wave_number  (wave_number) float64 1.278e+03 1.278e+03 ... 2.82e+03 2.82e+03

In [8]:
print(example_dataset1.isnull().sum().item())
print(example_dataset2.isnull().sum().item())
print(example_dataset_sum.isnull().sum().item())

0
0
2466450


In [9]:
# Stack X_0 and X_2 into a single dimension
reshaped_data_array_1 = example_dataset1.stack(X_0_combined=("X_0", "X_2"))

# Remove the MultiIndex and make X_0_combined a standard dimension
reshaped_data_array_1 = reshaped_data_array_1.reset_index("X_0_combined")

# Reorder dimensions to ensure the desired shape
reshaped_data_array_1 = reshaped_data_array_1.transpose("X_0_combined", "X_1", "wave_number")

# Verify the final shape
print(reshaped_data_array_1.shape)  # Should be (57, 10, 1015)
print(reshaped_data_array_1)

(57, 10, 1015)
<xarray.DataArray (X_0_combined: 57, X_1: 10, wave_number: 1015)>
array([[[ 74.983772,  91.177582,  85.314659, ...,  29.386446,
          20.207445,  16.533365],
        [ 92.627014,  80.883339,  85.314659, ...,  20.203182,
          31.229689,  25.718567],
        [ 82.335121,  98.530609,  88.256546, ...,  36.733059,
          25.718567,  18.370405],
        ...,
        [ 94.097282,  73.530304,  95.611259, ...,  25.71314 ,
          23.881527,  34.90377 ],
        [ 85.275658,  73.530304,  82.372772, ...,  31.223101,
          23.881527,  25.718567],
        [102.9189  ,  83.824547,  88.256546, ...,  22.039835,
          31.229689,  22.044485]],

       [[ 73.513504,  73.530304,  73.547119, ...,  27.549793,
          33.06673 ,  22.044485],
        [ 76.454041,  98.530609,  79.430893, ...,  27.549793,
          29.392649,  36.74081 ],
        [ 77.924309,  73.530304,  83.843719, ...,  27.549793,
          34.90377 ,  25.718567],
...
        [ 61.751339,  83.824547,  80